In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 5
T = 8
d_k = 4
w = 4 
Q = torch.randint(0,2,(B,T,d_k),dtype = float)
K = torch.randint(0,2,(B,T,d_k),dtype = float)
V = torch.randint(0,2,(B,T,d_k),dtype = float)
mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

In [53]:
Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (B,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)]  #K_chunks.T(-2,-1) : (B,d_k,w) 
# @ = ( B,w,d_k) @ (B,d_k,w) -> (B,w,w) @(B,w,d_k) -> (B,w,d_k)
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]
chunk_curr = tuple([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i]) for i in range(len(Q_chunks))])
res = torch.cat(chunk_curr,dim=1).nan_to_num(0) 
res.shape

RuntimeError: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1

In [ ]:
print(Q)

tensor([[[0., 1., 0., 0.],
         [1., 0., 1., 0.],
         [1., 1., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 1.],
         [1., 1., 1., 1.],
         [0., 0., 0., 1.],
         [1., 0., 1., 0.]],

        [[1., 0., 0., 1.],
         [0., 0., 1., 1.],
         [1., 1., 1., 1.],
         [0., 0., 1., 1.],
         [1., 0., 0., 1.],
         [0., 0., 0., 0.],
         [1., 1., 1., 1.],
         [1., 0., 0., 1.]],

        [[0., 0., 0., 1.],
         [0., 0., 1., 0.],
         [1., 0., 1., 1.],
         [0., 1., 1., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 1.],
         [1., 0., 0., 1.],
         [1., 0., 1., 0.]],

        [[1., 0., 0., 1.],
         [0., 0., 1., 1.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [1., 0., 1., 1.],
         [0., 1., 0., 1.],
         [1., 0., 0., 0.],
         [1., 1., 1., 1.]],

        [[0., 0., 1., 0.],
         [1., 0., 1., 1.],
         [1., 1., 0., 0.],
         [1., 0., 1., 0.],
         [1., 1., 0.

In [ ]:
#get [-x2,x1] from [x1,x2]
Q = torch.reshape(Q,(B,T,d_k//2,2))
x1 = Q[:,:,:,0]
x2  = Q[:,:,:,1]
res_x = torch.stack((-x2,x1),dim=-1).reshape(B,T,d_k)
Q = torch.reshape(Q,(B,T,d_k))

theta = torch.tensor([pow(10000,(-2*t)/d_k) for t in range(d_k//2)])
m = torch.arange(T).unsqueeze(1)
m_theta = theta*m
cos_theta = torch.cos(m_theta).repeat_interleave(2,dim=-1)
sin_theta = torch.sin(m_theta).repeat_interleave(2,dim=-1)

res = res_x*sin_theta + Q*cos_theta
res.shape


torch.Size([5, 8, 4])

In [ ]:
#get [-x2,x1] from [x1,x2]
x = torch.reshape(x,(B,T,d_k//2,2))
x1 = x[:,:,:,0]
x2  = x[:,:,:,1]
res_x = torch.stack((-x2,x1),dim=-1).reshape(B,T,d_k)

theta = torch.tensor([pow(10000,(-2*t)/d_k) for t in range(d_k//2)])
m = torch.arange(T).unsqueeze(1)
m_theta = theta*m
cos_theta = torch.cos(m_theta).repeat_interleave(2,dim=-1)
sin_theta = torch.sin(m_theta).repeat_interleave(2,dim=-1)

res = res_x*sin_theta + x*cos_theta
res.shape


RuntimeError: The size of tensor a (32) must match the size of tensor b (4) at non-singleton dimension 1

In [121]:
sin_theta1

tensor([[ 0.0000,  0.0000],
        [ 0.8415,  0.0100],
        [ 0.9093,  0.0200],
        [ 0.1411,  0.0300],
        [-0.7568,  0.0400],
        [-0.9589,  0.0500],
        [-0.2794,  0.0600],
        [ 0.6570,  0.0699]])

In [ ]:
q_pos = torch.arange(T).view(T,1)
k_pos = torch.arange(T).view(1,T)
k_pos-q_pos


tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [-1,  0,  1,  2,  3,  4,  5,  6],
        [-2, -1,  0,  1,  2,  3,  4,  5],
        [-3, -2, -1,  0,  1,  2,  3,  4],
        [-4, -3, -2, -1,  0,  1,  2,  3],
        [-5, -4, -3, -2, -1,  0,  1,  2],
        [-6, -5, -4, -3, -2, -1,  0,  1],
        [-7, -6, -5, -4, -3, -2, -1,  0]])

In [ ]:
n_heads = 8
slopes = torch.tensor([pow(2,-8/n_heads)**i for i in range(1,n_heads+1)],dtype=float)

tensor([[ 0.0000,  0.5000,  1.0000,  1.5000,  2.0000,  2.5000,  3.0000,  3.5000],
        [-0.5000,  0.0000,  0.5000,  1.0000,  1.5000,  2.0000,  2.5000,  3.0000],
        [-1.0000, -0.5000,  0.0000,  0.5000,  1.0000,  1.5000,  2.0000,  2.5000],
        [-1.5000, -1.0000, -0.5000,  0.0000,  0.5000,  1.0000,  1.5000,  2.0000],
        [-2.0000, -1.5000, -1.0000, -0.5000,  0.0000,  0.5000,  1.0000,  1.5000],
        [-2.5000, -2.0000, -1.5000, -1.0000, -0.5000,  0.0000,  0.5000,  1.0000],
        [-3.0000, -2.5000, -2.0000, -1.5000, -1.0000, -0.5000,  0.0000,  0.5000],
        [-3.5000, -3.0000, -2.5000, -2.0000, -1.5000, -1.0000, -0.5000,  0.0000]],
       dtype=torch.float64)